## Reading the existing processed file for each dataset

In [1]:
import os
import copy
import sys
import argparse
import asyncio
from tqdm import tqdm
import random
from pprint import pprint
from dotenv import load_dotenv
from collections import defaultdict

# Ensure repository root is on sys.path so "Code" is importable when run directly
REPO_ROOT = '/dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA'
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
# Load environment variables from .env file
load_dotenv(os.path.join(REPO_ROOT, ".env"))

from Code.src.utils.io import read_json_file, save_json_file
from Code.src.utils.qg_and_pd_utils import read_data_split

/dartfs-hpc/rc/home/j/f006f3j/lab/omar/conda/loqaGpu/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
import os
import json
import random


def is_valid_sample(sample):
    """Check if a sample is valid (has non-null, non-empty raw-initial-ground-truth)."""
    arg_value = sample.get("raw-initial-ground-truth", None)
    if arg_value is None:
        return False
    if isinstance(arg_value, (list, str)) and len(arg_value) == 0:
        return False
    if isinstance(arg_value, list) and len(arg_value) == 1 and arg_value[0] == "null":
        return False
    return True


def filter_valid_samples(samples):
    """Filter out invalid samples (where raw-initial-ground-truth is None, empty, or "null")."""
    return [sample for sample in samples if is_valid_sample(sample)]


def load_all_splits_from_processed(dataset_root: str, dataset_name: str):
    """Load all samples from train/dev/test splits in the *-processed.json file."""
    processed_path = os.path.join(dataset_root, dataset_name, f"{dataset_name}-all.json")
    data = read_json_file(processed_path)

    all_samples = []
    for split_name in ["train", "dev", "test"]:
        if split_name in data:
            all_samples.extend(data[split_name])
    return all_samples


def dedup_by_context_role_gt(samples):
    """Ensure no duplicate (context, role, raw-initial-ground-truth) appears more than once."""
    seen = set()
    unique = []
    for s in samples:
        ctx = s.get("context")
        role = s.get("role")
        gt = tuple(s.get("raw-initial-ground-truth") or [])
        key = (ctx, role, gt)
        if key in seen:
            continue
        seen.add(key)
        unique.append(s)
    return unique


def split_samples_for_dataset(dataset_name: str, samples):
    """Shuffle and split into train/dev/test with dataset-specific counts."""
    # CaseReportBench: dev=350, test=350; others: dev=500, test=500
    if dataset_name == "CaseReportBench":
        test_count = 350
        dev_count = 350
    else:
        test_count = 500
        dev_count = 500

    random.shuffle(samples)
    n = len(samples)
    if n < test_count + dev_count:
        raise ValueError(
            f"Not enough valid samples for {dataset_name}: "
            f"have {n}, need at least {test_count + dev_count} for test+dev."
        )

    test_split = samples[:test_count]
    dev_split = samples[test_count:test_count + dev_count]
    train_split = samples[test_count + dev_count:]
    return {"train": train_split, "dev": dev_split, "test": test_split}


def create_final_splits_for_dataset(dataset_root: str, dataset_name: str, seed: int = 42):
    """High-level helper to create and save final train/dev/test splits."""
    random.seed(seed)

    # 1) Load all splits from processed
    all_samples = load_all_splits_from_processed(dataset_root, dataset_name)

    # 2) Filter valid samples
    valid_samples = filter_valid_samples(all_samples)

    # 3) Deduplicate by (context, role, raw-initial-ground-truth)
    unique_samples = dedup_by_context_role_gt(valid_samples)

    print(
        f"{dataset_name}: {len(all_samples)} total, "
        f"{len(valid_samples)} valid, {len(unique_samples)} unique (by context, role, GT)."
    )

    # 4) Split into train/dev/test with requested counts
    splits = split_samples_for_dataset(dataset_name, unique_samples)
    for split_name, split_samples in splits.items():
        print(f"{split_name}: {len(split_samples)} samples")

    # 5) Save each split to its own file and assign serial-numbers
    for split_name, split_samples in splits.items():
        # Reassign serial-number as split_name-idx (1-based)
        for idx, sample in enumerate(split_samples, start=1):
            sample["serial-number"] = f"{split_name}-{idx}"

        out_path = os.path.join(dataset_root, dataset_name, f"{dataset_name}-{split_name}.json")
        save_json_file(split_samples, out_path)
        print(f"Saved {split_name} split ({len(split_samples)} samples) to {out_path}")

    return splits

dataset_root = '/dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset'
dataset_names = ["DiscourseEE", "PHEE", "CaseReportBench", "MACCROBAT"]
seed = 42
for dataset_name in dataset_names:
    print(f"{'='*60}")
    print(f"Creating final splits for {dataset_name}")
    create_final_splits_for_dataset(dataset_root, dataset_name, seed)

Creating final splits for DiscourseEE
Reading JSON file from: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/DiscourseEE/DiscourseEE-all.json
DiscourseEE: 7446 total, 3354 valid, 3222 unique (by context, role, GT).
train: 2222 samples
dev: 500 samples
test: 500 samples
Data saved to: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/DiscourseEE/DiscourseEE-train.json
Saved train split (2222 samples) to /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/DiscourseEE/DiscourseEE-train.json
Data saved to: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/DiscourseEE/DiscourseEE-dev.json
Saved dev split (500 samples) to /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/DiscourseEE/DiscourseEE-dev.json
Data saved to: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/DiscourseEE/DiscourseEE-test.json
Saved test split (500 samples) to /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/DiscourseEE/DiscourseEE-test.json
Creating final splits for PHEE
Reading JSON file from: /dartf

In [10]:
for dataset_name in dataset_names:
    print(f"{'='*60}")
    print("Reading processed data for", dataset_name)
    for split_name in ["train", "dev", "test"]:
        file_path = os.path.join(dataset_root, dataset_name, f"{dataset_name}-{split_name}.json")
        if os.path.exists(file_path):
            print("Size of", split_name, "split:", len(read_json_file(file_path)))
        else:
            print(f"Warning: {file_path} not found. Skipping {split_name} split.")


Reading processed data for DiscourseEE
Reading JSON file from: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/DiscourseEE/DiscourseEE-train.json
Size of train split: 2222
Reading JSON file from: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/DiscourseEE/DiscourseEE-dev.json
Size of dev split: 500
Reading JSON file from: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/DiscourseEE/DiscourseEE-test.json
Size of test split: 500
Reading processed data for PHEE
Reading JSON file from: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/PHEE/PHEE-train.json
Size of train split: 21274
Reading JSON file from: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/PHEE/PHEE-dev.json
Size of dev split: 500
Reading JSON file from: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/PHEE/PHEE-test.json
Size of test split: 500
Reading processed data for CaseReportBench
Reading JSON file from: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/CaseReportBench/CaseReportBench-train.json
Si

## Statistics of the dataset

In [7]:
import numpy as np

dataset_root = '/dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset'
dataset_names = ["CaseReportBench", "PHEE", "DiscourseEE", "MACCROBAT"]
doc_id_keys = {
    "CaseReportBench": "pmcid",
    "PHEE": "id",
    "DiscourseEE": "doc_id",
    "MACCROBAT": "doc_id",
}

all_stats = {}

for dataset_name in dataset_names:
    samples = []
    for split_name in ["train", "dev", "test"]:
        file_path = os.path.join(dataset_root, dataset_name, f"{dataset_name}-{split_name}.json")
        if os.path.exists(file_path):
            samples.extend(read_json_file(file_path))

    # 1) Unique roles
    roles = set(s.get("role", "") for s in samples)

    # 2) Context length in words
    ctx_lengths = [len(s.get("context", "").split()) for s in samples]

    # 3) Argument counts (list length per sample)
    arg_counts = [len(s.get("raw-initial-ground-truth") or []) for s in samples]
    total_args = int(np.sum(arg_counts))

    # 4) Document-level argument density = total arguments / unique documents
    doc_id_key = doc_id_keys[dataset_name]
    unique_doc_ids = {
        s.get(doc_id_key)
        for s in samples
        if s.get(doc_id_key) is not None
    }
    num_unique_docs = len(unique_doc_ids)
    args_per_doc = (total_args / num_unique_docs) if num_unique_docs > 0 else 0

    # 5) Avg argument length in words (across all individual arguments)
    arg_word_lengths = []
    for s in samples:
        for arg in (s.get("raw-initial-ground-truth") or []):
            if isinstance(arg, str):
                arg_word_lengths.append(len(arg.split()))

    stats = {
        "Total samples": len(samples),
        "Unique docs": num_unique_docs,
        "Total arguments": total_args,
        "Unique roles": len(roles),
        "Role list": sorted(roles),
        "Avg context length (words)": np.mean(ctx_lengths),
        "Median context length (words)": np.median(ctx_lengths),
        "Min / Max context length": (min(ctx_lengths), max(ctx_lengths)),
        "Argument density (args/doc)": args_per_doc,
        "Avg argument length (words)": np.mean(arg_word_lengths) if arg_word_lengths else 0,
        "Median argument length (words)": np.median(arg_word_lengths) if arg_word_lengths else 0,
    }
    all_stats[dataset_name] = stats

    print(f"\n{'='*60}")
    print(f"  {dataset_name}")
    print(f"{'='*60}")
    for k, v in stats.items():
        if isinstance(v, float):
            print(f"  {k:40s}: {v:.2f}")
        else:
            print(f"  {k:40s}: {v}")

Reading JSON file from: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/CaseReportBench/CaseReportBench-train.json
Reading JSON file from: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/CaseReportBench/CaseReportBench-dev.json
Reading JSON file from: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/CaseReportBench/CaseReportBench-test.json

  CaseReportBench
  Total samples                           : 1320
  Unique docs                             : 138
  Total arguments                         : 3555
  Unique roles                            : 18
  Role list                               : ['Age-at-Presentation', 'Age-of-Onset', 'Cardiovascular-System', 'Confirmed-Diagnosis-IEM', 'Dermatology', 'Endocrinology', 'Eyes-Ears-Nose-Throat', 'Gastrointestinal-System', 'Genitourinary-System', 'IEM-Treatment', 'Laboratory-and-Imaging', 'Lymphatic-System', 'Musculoskeletal-System', 'Neurology', 'Patient-History', 'Pregnancy', 'Respiratory-System', 'Vitals-and-Hematology']
  Avg con

In [9]:
import pandas as pd

ordered_names = ["CaseReportBench", "PHEE", "DiscourseEE", "MACCROBAT"]
rows = []
for name in ordered_names:
    st = all_stats[name]
    rows.append({
        "Dataset": name,
        "Samples": st["Total samples"],
        "Unique Docs": st["Unique docs"],
        "Total Args": st["Total arguments"],
        "Unique Roles": st["Unique roles"],
        "Avg Ctx Len (words)": round(st["Avg context length (words)"], 1),
        "Med Ctx Len (words)": round(st["Median context length (words)"], 1),
        "Arg Density (args/doc)": round(st["Argument density (args/doc)"], 2),
        "Avg Arg Len (words)": round(st["Avg argument length (words)"], 2),
        "Med Arg Len (words)": round(st["Median argument length (words)"], 2),
    })

df = pd.DataFrame(rows)
df

,Dataset,Samples,Unique Docs,Total Args,Unique Roles,Avg Ctx Len (words),Med Ctx Len (words),Arg Density (args/doc),Avg Arg Len (words),Med Arg Len (words)
0,CaseReportBench,1320,138,3555,18,581.7,521.0,25.76,20.71,10.0
1,PHEE,22274,4827,24454,14,19.7,19.0,5.07,2.45,2.0
2,DiscourseEE,3222,392,3709,34,120.8,118.0,9.46,3.04,2.0
3,MACCROBAT,6882,200,8120,22,24.2,22.0,40.60,1.73,1.0


## Creating test splits for DocEE and GENEVA

In [8]:
import os
import sys
import json
import random

REPO_ROOT = '/dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA'
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from Code.src.utils.io import read_json_file, save_json_file


def is_valid_sample(sample):
    """Check if a sample is valid (has non-null, non-empty raw-initial-ground-truth)."""
    arg_value = sample.get("raw-initial-ground-truth", None)
    if arg_value is None:
        return False
    if isinstance(arg_value, (list, str)) and len(arg_value) == 0:
        return False
    if isinstance(arg_value, list) and len(arg_value) == 1 and arg_value[0] == "null":
        return False
    return True


def filter_valid_samples(samples):
    """Filter out invalid samples (where raw-initial-ground-truth is None, empty, or "null")."""
    return [sample for sample in samples if is_valid_sample(sample)]


def dedup_by_context_role_gt(samples):
    """Ensure no duplicate (context, role, raw-initial-ground-truth) appears more than once."""
    seen = set()
    unique = []
    for s in samples:
        ctx = s.get("context")
        role = s.get("role")
        gt = tuple(s.get("raw-initial-ground-truth") or [])
        key = (ctx, role, gt)
        if key in seen:
            continue
        seen.add(key)
        unique.append(s)
    return unique


def make_clean_role(event, role):
    """Build an LLM-friendly role name from event and role: 'event name: role name'."""
    def _normalize(s):
        return " ".join(s.replace("_", " ").replace(" - ", " ").replace("-", " ").split()).lower()
    return f"{_normalize(event)}: {_normalize(role)}"


dataset_root = '/dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset'
seed = 42
test_count = 500

datasets_config = {
    "DocEE": {
        "source_file": "DocEE-processed-test-data-EAE-Eval.json",
        "doc_id_key": "id",
    },
    "GENEVA": {
        "source_file": "GENEVA-processed-test-data-EAE-Eval.json",
        "doc_id_key": "id",
    },
}

random.seed(seed)

for dataset_name, config in datasets_config.items():
    print(f"\n{'='*60}")
    print(f"  Creating test split for {dataset_name}")
    print(f"{'='*60}")

    source_path = os.path.join(dataset_root, dataset_name, config["source_file"])
    all_samples = read_json_file(source_path)
    print(f"Loaded {len(all_samples)} total samples from {config['source_file']}")

    valid_samples = filter_valid_samples(all_samples)
    print(f"Valid samples: {len(valid_samples)}  (filtered out {len(all_samples) - len(valid_samples)} invalid)")

    unique_samples = dedup_by_context_role_gt(valid_samples)
    print(f"Unique samples (by context, role, GT): {len(unique_samples)}  (removed {len(valid_samples) - len(unique_samples)} duplicates)")

    doc_id_key = config["doc_id_key"]
    total_docs = {s.get(doc_id_key) for s in unique_samples if s.get(doc_id_key) is not None}
    print(f"Unique documents: {len(total_docs)}")

    if len(unique_samples) < test_count:
        print(f"WARNING: Only {len(unique_samples)} unique samples available, less than {test_count}. Using all.")
        test_count_actual = len(unique_samples)
    else:
        test_count_actual = test_count

    random.shuffle(unique_samples)
    test_samples = unique_samples[:test_count_actual]

    for idx, sample in enumerate(test_samples, start=1):
        sample["serial-number"] = f"test-{idx}"

    for sample in test_samples:
        sample["old-role"] = sample.get("role")
        event = sample.get("event", "")
        old_role = sample.get("role", "")
        if event and old_role:
            sample["role"] = make_clean_role(event, old_role)

    selected_docs = {s.get(doc_id_key) for s in test_samples if s.get(doc_id_key) is not None}
    print(f"\nSelected {len(test_samples)} test samples from {len(selected_docs)} unique documents (out of {len(total_docs)} total)")

    event_counts = {}
    for s in test_samples:
        ev = s.get("event", "unknown")
        event_counts[ev] = event_counts.get(ev, 0) + 1
    print(f"Event type distribution in test set ({len(event_counts)} types):")
    for ev, cnt in sorted(event_counts.items(), key=lambda x: -x[1]):
        print(f"  {ev:40s}: {cnt}")

    out_path = os.path.join(dataset_root, dataset_name, f"{dataset_name}-test.json")
    save_json_file(test_samples, out_path)
    print(f"\nSaved test split to: {out_path}")


  Creating test split for DocEE
Reading JSON file from: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/DocEE/DocEE-processed-test-data-EAE-Eval.json
Loaded 2368 total samples from DocEE-processed-test-data-EAE-Eval.json
Valid samples: 2368  (filtered out 0 invalid)
Unique samples (by context, role, GT): 2368  (removed 0 duplicates)
Unique documents: 500

Selected 500 test samples from 322 unique documents (out of 500 total)
Event type distribution in test set (53 types):
  air crash                               : 51
  armed conflict                          : 32
  sports competition                      : 29
  protest                                 : 28
  strike                                  : 21
  commitcrime - sentence                  : 18
  earthquakes                             : 17
  disease outbreaks                       : 16
  fire                                    : 15
  resignation_dismissal                   : 15
  diplomatic talks                        : 15
 

In [9]:
import os
import sys
import json
import numpy as np
import pandas as pd

REPO_ROOT = '/dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA'
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from Code.src.utils.io import read_json_file

dataset_root = '/dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset'

new_datasets = {
    "DocEE": {"doc_id_key": "id"},
    "GENEVA": {"doc_id_key": "id"},
}

all_stats = {}

for dataset_name, config in new_datasets.items():
    test_path = os.path.join(dataset_root, dataset_name, f"{dataset_name}-test.json")
    samples = read_json_file(test_path)
    doc_id_key = config["doc_id_key"]

    roles = set(s.get("role", "") for s in samples)
    events = set(s.get("event", "") for s in samples)

    ctx_lengths = [len(s.get("context", "").split()) for s in samples]

    arg_counts = [len(s.get("raw-initial-ground-truth") or []) for s in samples]
    total_args = int(np.sum(arg_counts))

    unique_doc_ids = {s.get(doc_id_key) for s in samples if s.get(doc_id_key) is not None}
    num_unique_docs = len(unique_doc_ids)
    args_per_doc = (total_args / num_unique_docs) if num_unique_docs > 0 else 0

    arg_word_lengths = []
    for s in samples:
        for arg in (s.get("raw-initial-ground-truth") or []):
            if isinstance(arg, str):
                arg_word_lengths.append(len(arg.split()))

    stats = {
        "Test samples": len(samples),
        "Unique docs": num_unique_docs,
        "Unique event types": len(events),
        "Total arguments": total_args,
        "Unique roles": len(roles),
        "Role list": sorted(roles),
        "Avg context length (words)": np.mean(ctx_lengths),
        "Median context length (words)": np.median(ctx_lengths),
        "Min / Max context length": (min(ctx_lengths), max(ctx_lengths)),
        "Argument density (args/doc)": args_per_doc,
        "Avg argument length (words)": np.mean(arg_word_lengths) if arg_word_lengths else 0,
        "Median argument length (words)": np.median(arg_word_lengths) if arg_word_lengths else 0,
    }
    all_stats[dataset_name] = stats

    print(f"\n{'='*60}")
    print(f"  {dataset_name} (test split)")
    print(f"{'='*60}")
    for k, v in stats.items():
        if isinstance(v, float):
            print(f"  {k:40s}: {v:.2f}")
        else:
            print(f"  {k:40s}: {v}")

rows = []
for name in ["DocEE", "GENEVA"]:
    st = all_stats[name]
    rows.append({
        "Dataset": name,
        "Test Samples": st["Test samples"],
        "Unique Docs": st["Unique docs"],
        "Event Types": st["Unique event types"],
        "Total Args": st["Total arguments"],
        "Unique Roles": st["Unique roles"],
        "Avg Ctx Len (words)": round(st["Avg context length (words)"], 1),
        "Med Ctx Len (words)": round(st["Median context length (words)"], 1),
        "Arg Density (args/doc)": round(st["Argument density (args/doc)"], 2),
        "Avg Arg Len (words)": round(st["Avg argument length (words)"], 2),
        "Med Arg Len (words)": round(st["Median argument length (words)"], 2),
    })

df = pd.DataFrame(rows)
df

Reading JSON file from: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/DocEE/DocEE-test.json

  DocEE (test split)
  Test samples                            : 500
  Unique docs                             : 322
  Unique event types                      : 53
  Total arguments                         : 749
  Unique roles                            : 238
  Role list                               : ['air crash: accident investigator', 'air crash: aircraft agency', 'air crash: casualties and losses', 'air crash: cause', 'air crash: crew', 'air crash: date', 'air crash: flight no.', 'air crash: location', 'air crash: passengers', 'air crash: scheduled landing place', 'air crash: survivors', 'appoint inauguration: appointee', 'appoint inauguration: inauguration time', 'appoint inauguration: last job of the appointee', 'appoint inauguration: position', 'appoint inauguration: predecessor', 'armed conflict: attacker', 'armed conflict: casualties and losses', 'armed conflict: damaged facilit

,Dataset,Test Samples,Unique Docs,Event Types,Total Args,Unique Roles,Avg Ctx Len (words),Med Ctx Len (words),Arg Density (args/doc),Avg Arg Len (words),Med Arg Len (words)
0,DocEE,500,322,53,749,238,643.2,529.5,2.33,4.00,3.0
1,GENEVA,500,74,108,538,218,30.9,29.0,7.27,5.24,3.0


## previously used this code


## Preparing the gold test data

In [9]:
def is_valid_sample(sample):
    """Check if a sample is valid (has non-null, non-empty raw-initial-ground-truth)."""
    arg_value = sample.get("raw-initial-ground-truth", None)
    if arg_value is None:
        return False
    if isinstance(arg_value, (list, str)) and len(arg_value) == 0:
        return False
    if isinstance(arg_value, list) and len(arg_value) == 1 and arg_value[0] == "null":
        return False
    return True


def filter_valid_samples(samples):
    """Filter out invalid samples (where raw-initial-ground-truth is None, empty, or "null")."""
    return [sample for sample in samples if is_valid_sample(sample)]


def get_unique_documents(samples, doc_id_key):
    """Get set of unique document IDs from samples."""
    return {sample.get(doc_id_key) for sample in samples if sample.get(doc_id_key) is not None}


def create_gold_test_set(processed_data, dataset_name, split_name, sample_count=500):
    """
    Create a gold set based on the split name by selecting samples only from that split.
    
    Args:
        processed_data: Dictionary with split keys ('train', 'dev', 'test')
        dataset_name: Name of the dataset ('DiscourseEE', 'PHEE', or 'CaseReportBench')
        split_name: Name of the split to use ('train', 'dev', or 'test')
        sample_count: Number of samples to select
    
    Returns:
        List of selected samples
    """
    # Determine document ID key
    doc_id_keys = {
        "DiscourseEE": "doc_id",
        "PHEE": "id",
        "CaseReportBench": "pmcid",
        "MACCROBAT": "doc_id"
    }
    doc_id_key = doc_id_keys.get(dataset_name)
    if doc_id_key is None:
        raise ValueError(f"Unknown dataset: {dataset_name}")
    
    # Get samples only from the specified split
    all_samples = processed_data.get(split_name, [])
    if not all_samples:
        raise ValueError(f"Split '{split_name}' not found in processed_data")
    
    print(f"\nTotal {split_name} samples available: {len(all_samples)}")
    
    # Filter valid samples
    valid_samples = filter_valid_samples(all_samples)
    invalid_count = len(all_samples) - len(valid_samples)
    print(f"Valid {split_name} samples: {len(valid_samples)}")
    print(f"Invalid {split_name} samples filtered out: {invalid_count}")
    
    # Calculate total unique documents
    total_docs = get_unique_documents(valid_samples, doc_id_key)
    print(f"Total unique documents in {split_name} split: {len(total_docs)}")
    
    # Randomly sample samples
    print(f"\nRandomly selecting {sample_count} samples from {split_name} split for gold set...")
    random.shuffle(valid_samples)
    selected_samples = valid_samples[:sample_count]
    
    # Calculate document coverage
    selected_doc_ids = get_unique_documents(selected_samples, doc_id_key)
    print(f"Selected {len(selected_samples)} samples from {len(selected_doc_ids)} unique documents (out of {len(total_docs)} total documents)")
    
    return selected_samples

In [ ]:
# # Set random seed for reproducibility
# random.seed(42)
# data_path = '/dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset'
# # Process each dataset to create gold test sets
# datasets_to_process = ["DiscourseEE", "PHEE", "CaseReportBench", "MACCROBAT"]

# for dataset_name in datasets_to_process:
#     print(f"\n{'='*60}")
#     print(f"Creating gold test set for {dataset_name}")
#     print(f"{'='*60}")
    
#     # Read processed data
#     dataset_dir = os.path.join(data_path, dataset_name)
#     dataset_split_path = os.path.join(dataset_dir, f"{dataset_name}-final.json")
    
#     if not os.path.exists(dataset_split_path):
#         print(f"Warning: {dataset_split_path} not found. Skipping {dataset_name}.")
#         continue
    
#     processed_data = read_json_file(dataset_split_path)
#     print(f"Loaded processed data from: {dataset_split_path}")
    
#     # Create gold test set from test split
#     split_name = 'test'
#     test_samples = create_gold_test_set(processed_data, dataset_name, split_name, sample_count=500)
    
#     # Save gold test set
#     final_output_path = os.path.join(dataset_dir, f"{dataset_name}-gold-test.json")
#     save_json_file({'test': test_samples}, final_output_path)
    
#     print(f"\nGold test set saved to: {final_output_path}")
#     print(f"Test samples: {len(test_samples)}")


Creating gold test set for DiscourseEE
Reading JSON file from: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/DiscourseEE/DiscourseEE-final.json
Loaded processed data from: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/DiscourseEE/DiscourseEE-final.json

Total test samples available: 1883
Valid test samples: 874
Invalid test samples filtered out: 1009
Total unique documents in test split: 99

Randomly selecting 500 samples from test split for gold set...
Selected 500 samples from 97 unique documents (out of 99 total documents)
Data saved to: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/DiscourseEE/DiscourseEE-gold-test.json

Gold test set saved to: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/DiscourseEE/DiscourseEE-gold-test.json
Test samples: 500

Creating gold test set for PHEE
Reading JSON file from: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/PHEE/PHEE-final.json
Loaded processed data from: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/PHEE/

## Macrobat Dataset Merging

In [ ]:
# # Load the processed data and save MACCROBAT dataset
# split_names = ["train", "dev", "test"]
# dataset_name = "MACCROBAT"
# data_path = '/dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset'

# dataset_dir = os.path.join(data_path, dataset_name)
# final_dataset = {}

# for split_name in split_names:
#     dataset_split_path = os.path.join(dataset_dir, f"{dataset_name}-{split_name}.json")

#     if not os.path.exists(dataset_split_path):
#         print(f"Warning: {dataset_split_path} not found. Skipping {split_name}.")
#         continue

#     final_dataset[split_name] = read_json_file(dataset_split_path)
#     print(f"Loaded {split_name}: {len(final_dataset[split_name])} samples")

# output_path = os.path.join(dataset_dir, f"{dataset_name}-final.json")
# save_json_file(final_dataset, output_path)
# print(f"\nMACCROBAT dataset saved to: {output_path}")
# print(f"Train: {len(final_dataset.get('train', []))} | Dev: {len(final_dataset.get('dev', []))} | Test: {len(final_dataset.get('test', []))}")

Reading JSON file from: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/MACCROBAT/MACCROBAT-train.json
Loaded train: 5508 samples
Reading JSON file from: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/MACCROBAT/MACCROBAT-dev.json
Loaded dev: 671 samples
Reading JSON file from: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/MACCROBAT/MACCROBAT-test.json
Loaded test: 741 samples
Data saved to: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/MACCROBAT/MACCROBAT-final.json

MACCROBAT dataset saved to: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/MACCROBAT/MACCROBAT-final.json
Train: 5508 | Dev: 671 | Test: 741


## CaseReportBench Spit Creation

In [26]:
# def create_splits_casereportbench(processed_data, test_count=500):
#     """
#     Create new splits for CaseReportBench dataset.
    
#     Args:
#         processed_data: Dictionary with 'test' key (all data)
#         test_count: Number of samples for test set
    
#     Returns:
#         Dictionary with new 'train', 'dev', 'test' splits
#     """
#     doc_id_key = "pmcid"
    
#     # Get all samples
#     all_samples = processed_data.get('test', [])
#     print(f"\nTotal samples available: {len(all_samples)}")
    
#     # Filter valid samples only
#     valid_samples = filter_valid_samples(all_samples)
#     invalid_count = len(all_samples) - len(valid_samples)
#     print(f"Valid samples: {len(valid_samples)}")
#     print(f"Invalid samples filtered out: {invalid_count}")
    
#     # Calculate total unique documents
#     total_docs = set(sample.get(doc_id_key) for sample in valid_samples if sample.get(doc_id_key) is not None)
#     print(f"Total unique documents: {len(total_docs)}")
    
#     # Shuffle all valid samples
#     random.shuffle(valid_samples)
    
#     # Step 1: Randomly select test set (500 samples)
#     print(f"\nRandomly selecting {test_count} samples for test set...")
#     test_samples = valid_samples[:test_count]
#     remaining_samples = valid_samples[test_count:]
    
#     # Calculate document coverage for test
#     test_doc_ids = set(sample.get(doc_id_key) for sample in test_samples if sample.get(doc_id_key) is not None)
#     print(f"Test set: {len(test_samples)} samples from {len(test_doc_ids)} unique documents (out of {len(total_docs)} total documents)")
    
#     # Step 2: Remaining data split into train (80%) and dev (20%)
#     print(f"\nRemaining samples: {len(remaining_samples)}")
    
#     # Split: 80% train, 20% dev
#     train_size = int(len(remaining_samples) * 0.8)
#     train_samples = remaining_samples[:train_size]
#     dev_samples = remaining_samples[train_size:]
    
#     # Calculate unique documents covered for train and dev
#     train_doc_ids = set(sample.get(doc_id_key) for sample in train_samples if sample.get(doc_id_key) is not None)
#     dev_doc_ids = set(sample.get(doc_id_key) for sample in dev_samples if sample.get(doc_id_key) is not None)
    
#     print(f"Train set: {len(train_samples)} samples from {len(train_doc_ids)} unique documents (out of {len(total_docs)} total documents)")
#     print(f"Dev set: {len(dev_samples)} samples from {len(dev_doc_ids)} unique documents (out of {len(total_docs)} total documents)")
    
#     # Update serial numbers
#     for idx, sample in enumerate(test_samples, 1):
#         sample['serial-number'] = f"test-{idx}"
#     for idx, sample in enumerate(dev_samples, 1):
#         sample['serial-number'] = f"dev-{idx}"
#     for idx, sample in enumerate(train_samples, 1):
#         sample['serial-number'] = f"train-{idx}"
    
#     return {
#         'train': train_samples,
#         'dev': dev_samples,
#         'test': test_samples
#     }

# # Set random seed for reproducibility
# random.seed(42)
# data_path = '/dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset'
# # Process each dataset to create gold test sets
# datasets_to_process = ["CaseReportBench"]

# for dataset_name in datasets_to_process:
#     print(f"\n{'='*60}")
#     print(f"Creating splits for {dataset_name}")
#     print(f"{'='*60}")
    
#     # Read processed data
#     dataset_dir = os.path.join(data_path, dataset_name)
#     dataset_split_path = os.path.join(dataset_dir, f"{dataset_name}-processed.json")
    
#     if not os.path.exists(dataset_split_path):
#         print(f"Warning: {dataset_split_path} not found. Skipping {dataset_name}.")
#         continue
    
#     processed_data = read_json_file(dataset_split_path)
#     print(f"Loaded processed data from: {dataset_split_path}")
    
#     # Create gold test set from test split
#     new_splits = create_splits_casereportbench(processed_data, test_count=500)
    
#     # Save gold test set
#     final_output_path = os.path.join(dataset_dir, f"{dataset_name}-final.json")
#     save_json_file({'test': test_samples}, final_output_path)
    
#     print(f"Test samples: {len(test_samples)}")


Creating splits for CaseReportBench
Reading JSON file from: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/CaseReportBench/CaseReportBench-processed.json
Loaded processed data from: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/CaseReportBench/CaseReportBench-processed.json

Total samples available: 2484
Valid samples: 1320
Invalid samples filtered out: 1164
Total unique documents: 138

Randomly selecting 500 samples for test set...
Test set: 500 samples from 138 unique documents (out of 138 total documents)

Remaining samples: 820
Train set: 656 samples from 138 unique documents (out of 138 total documents)
Dev set: 164 samples from 97 unique documents (out of 138 total documents)
Data saved to: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/CaseReportBench/CaseReportBench-final.json

Gold test set saved to: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/CaseReportBench/CaseReportBench-final.json
Test samples: 500


## This will be useful in the future for creating train and dev samples 

In [ ]:
# def is_valid_sample(sample):
#     """
#     Check if a sample is valid (has non-null, non-empty raw-initial-ground-truth).
    
#     Args:
#         sample: Sample dictionary
    
#     Returns:
#         True if sample is valid, False otherwise
#     """
#     arg_value = sample.get("raw-initial-ground-truth", None)
#     if arg_value is None:
#         return False
#     if isinstance(arg_value, (list, str)) and len(arg_value) == 0:
#         return False
#     if isinstance(arg_value, list) and len(arg_value) == 1 and arg_value[0] == "null":
#         return False
#     return True


# def filter_valid_samples(samples):
#     """
#     Filter out invalid samples (where raw-initial-ground-truth is None, empty, or "null").
    
#     Args:
#         samples: List of sample dictionaries
    
#     Returns:
#         List of valid samples
#     """
#     return [sample for sample in samples if is_valid_sample(sample)]


# def group_samples_by_document(samples, doc_id_key):
#     """
#     Group samples by document ID to maximize document coverage.
#     Only groups valid samples (filters out invalid ones).
    
#     Args:
#         samples: List of sample dictionaries
#         doc_id_key: Key to use for document identification ('doc_id', 'id', or 'pmcid')
    
#     Returns:
#         Dictionary mapping document IDs to lists of valid samples
#     """
#     doc_groups = defaultdict(list)
#     for sample in samples:
#         # Only group valid samples
#         if is_valid_sample(sample):
#             doc_id = sample.get(doc_id_key)
#             if doc_id is not None:
#                 doc_groups[doc_id].append(sample)
#     return doc_groups


# def select_samples_maximizing_document_coverage(doc_groups, target_count):
#     """
#     Select samples to maximize the number of unique documents covered.
    
#     Strategy:
#     1. Sort documents by number of samples (ascending) to prioritize documents with fewer samples
#     2. Iteratively add documents until we reach the target count
#     3. If we exceed target, randomly sample from the last document added
    
#     Args:
#         doc_groups: Dictionary mapping document IDs to lists of samples
#         target_count: Target number of samples to select
    
#     Returns:
#         Tuple of (selected_samples, selected_doc_ids, remaining_doc_groups)
#         where remaining_doc_groups has the selected samples removed
#     """
#     # Create a deep copy to avoid modifying original
#     remaining_doc_groups = {k: v.copy() for k, v in doc_groups.items()}
    
#     # Sort documents by number of samples (ascending)
#     sorted_docs = sorted(remaining_doc_groups.items(), key=lambda x: len(x[1]))
    
#     selected_samples = []
#     selected_doc_ids = set()
    
#     # First pass: add entire documents until we approach target
#     for doc_id, samples in sorted_docs:
#         if len(selected_samples) + len(samples) <= target_count:
#             selected_samples.extend(samples)
#             selected_doc_ids.add(doc_id)
#             # Remove all samples from this document
#             remaining_doc_groups[doc_id] = []
#         elif len(selected_samples) < target_count:
#             # We need some samples from this document but not all
#             remaining_needed = target_count - len(selected_samples)
#             # Randomly sample from this document (shuffle a copy to avoid modifying original)
#             samples_copy = samples.copy()
#             random.shuffle(samples_copy)
#             selected_samples.extend(samples_copy[:remaining_needed])
#             selected_doc_ids.add(doc_id)
#             # Remove selected samples from remaining
#             remaining_doc_groups[doc_id] = samples_copy[remaining_needed:]
#             break
    
#     # If we haven't reached target, add more documents
#     if len(selected_samples) < target_count:
#         sorted_docs = sorted(remaining_doc_groups.items(), key=lambda x: len(x[1]) if x[1] else 0)
#         for doc_id, samples in sorted_docs:
#             if not samples:  # Skip empty documents
#                 continue
#             if doc_id not in selected_doc_ids:
#                 remaining_needed = target_count - len(selected_samples)
#                 if remaining_needed <= 0:
#                     break
#                 # Add samples from this document (shuffle a copy to avoid modifying original)
#                 samples_copy = samples.copy()
#                 random.shuffle(samples_copy)
#                 num_to_take = min(remaining_needed, len(samples_copy))
#                 selected_samples.extend(samples_copy[:num_to_take])
#                 selected_doc_ids.add(doc_id)
#                 # Remove selected samples from remaining
#                 remaining_doc_groups[doc_id] = samples_copy[num_to_take:]
#                 if len(selected_samples) >= target_count:
#                     break
    
#     # Trim to exact target if we exceeded
#     if len(selected_samples) > target_count:
#         selected_samples = selected_samples[:target_count]
    
#     # Final validation: ensure all selected samples are valid
#     selected_samples = [s for s in selected_samples if is_valid_sample(s)]
    
#     # Clean up empty documents from remaining
#     remaining_doc_groups = {k: v for k, v in remaining_doc_groups.items() if v}
    
#     return selected_samples, selected_doc_ids, remaining_doc_groups


# def create_splits_discourseee_phee(processed_data, dataset_name, test_count=500, train_count=1500, dev_count=500):
#     """
#     Create new splits for DiscourseEE or PHEE datasets.
    
#     Args:
#         processed_data: Dictionary with 'train', 'dev', 'test' keys
#         dataset_name: Name of the dataset ('DiscourseEE' or 'PHEE')
#         test_count: Number of samples for test set
#         train_count: Number of samples for train set
#         dev_count: Number of samples for dev set
    
#     Returns:
#         Dictionary with new 'train', 'dev', 'test' splits
#     """
#     # Determine document ID key
#     if dataset_name == "DiscourseEE":
#         doc_id_key = "doc_id"
#     elif dataset_name == "PHEE":
#         doc_id_key = "id"
#     else:
#         raise ValueError(f"Unknown dataset: {dataset_name}")
    
#     # Process each split independently
#     new_splits = {}
    
#     # Process train split
#     if 'train' in processed_data:
#         train_samples_raw = processed_data['train']
#         print(f"\n{'='*60}")
#         print(f"Processing TRAIN split")
#         print(f"{'='*60}")
#         print(f"Total train samples available: {len(train_samples_raw)}")
        
#         # Filter valid samples
#         valid_train = filter_valid_samples(train_samples_raw)
#         invalid_train = len(train_samples_raw) - len(valid_train)
#         print(f"Valid train samples: {len(valid_train)}")
#         print(f"Invalid train samples filtered out: {invalid_train}")
        
#         # Calculate total unique documents in train split
#         total_train_docs = set(sample.get(doc_id_key) for sample in valid_train if sample.get(doc_id_key) is not None)
#         print(f"Total unique documents in train split: {len(total_train_docs)}")
        
#         # Randomly sample train samples
#         print(f"\nRandomly selecting {train_count} samples from TRAIN split...")
#         random.shuffle(valid_train)
#         train_samples = valid_train[:train_count]
        
#         # Calculate document coverage
#         train_doc_ids = set(sample.get(doc_id_key) for sample in train_samples if sample.get(doc_id_key) is not None)
#         print(f"Selected {len(train_samples)} train samples from {len(train_doc_ids)} unique documents (out of {len(total_train_docs)} total documents)")
#         new_splits['train'] = train_samples
#     else:
#         new_splits['train'] = []
    
#     # Process dev split
#     if 'dev' in processed_data:
#         dev_samples_raw = processed_data['dev']
#         print(f"\n{'='*60}")
#         print(f"Processing DEV split")
#         print(f"{'='*60}")
#         print(f"Total dev samples available: {len(dev_samples_raw)}")
        
#         # Filter valid samples
#         valid_dev = filter_valid_samples(dev_samples_raw)
#         invalid_dev = len(dev_samples_raw) - len(valid_dev)
#         print(f"Valid dev samples: {len(valid_dev)}")
#         print(f"Invalid dev samples filtered out: {invalid_dev}")
        
#         # Calculate total unique documents in dev split
#         total_dev_docs = set(sample.get(doc_id_key) for sample in valid_dev if sample.get(doc_id_key) is not None)
#         print(f"Total unique documents in dev split: {len(total_dev_docs)}")
        
#         # Randomly sample dev samples
#         print(f"\nRandomly selecting {dev_count} samples from DEV split...")
#         random.shuffle(valid_dev)
#         dev_samples = valid_dev[:dev_count]
        
#         # Calculate document coverage
#         dev_doc_ids = set(sample.get(doc_id_key) for sample in dev_samples if sample.get(doc_id_key) is not None)
#         print(f"Selected {len(dev_samples)} dev samples from {len(dev_doc_ids)} unique documents (out of {len(total_dev_docs)} total documents)")
#         new_splits['dev'] = dev_samples
#     else:
#         new_splits['dev'] = []
    
#     # Process test split
#     if 'test' in processed_data:
#         test_samples_raw = processed_data['test']
#         print(f"\n{'='*60}")
#         print(f"Processing TEST split")
#         print(f"{'='*60}")
#         print(f"Total test samples available: {len(test_samples_raw)}")
        
#         # Filter valid samples
#         valid_test = filter_valid_samples(test_samples_raw)
#         invalid_test = len(test_samples_raw) - len(valid_test)
#         print(f"Valid test samples: {len(valid_test)}")
#         print(f"Invalid test samples filtered out: {invalid_test}")
        
#         # Calculate total unique documents in test split
#         total_test_docs = set(sample.get(doc_id_key) for sample in valid_test if sample.get(doc_id_key) is not None)
#         print(f"Total unique documents in test split: {len(total_test_docs)}")
        
#         # Randomly sample test samples
#         print(f"\nRandomly selecting {test_count} samples from TEST split...")
#         random.shuffle(valid_test)
#         test_samples = valid_test[:test_count]
        
#         # Calculate document coverage
#         test_doc_ids = set(sample.get(doc_id_key) for sample in test_samples if sample.get(doc_id_key) is not None)
#         print(f"Selected {len(test_samples)} test samples from {len(test_doc_ids)} unique documents (out of {len(total_test_docs)} total documents)")
#         new_splits['test'] = test_samples
#     else:
#         new_splits['test'] = []
    
#     # Update serial numbers
#     for idx, sample in enumerate(new_splits.get('test', []), 1):
#         sample['serial-number'] = f"test-{idx}"
#     for idx, sample in enumerate(new_splits.get('dev', []), 1):
#         sample['serial-number'] = f"dev-{idx}"
#     for idx, sample in enumerate(new_splits.get('train', []), 1):
#         sample['serial-number'] = f"train-{idx}"
    
#     return new_splits


# def create_splits_casereportbench(processed_data, test_count=500):
#     """
#     Create new splits for CaseReportBench dataset.
    
#     Args:
#         processed_data: Dictionary with 'test' key (all data)
#         test_count: Number of samples for test set
    
#     Returns:
#         Dictionary with new 'train', 'dev', 'test' splits
#     """
#     doc_id_key = "pmcid"
    
#     # Get all samples
#     all_samples = processed_data.get('test', [])
#     print(f"\nTotal samples available: {len(all_samples)}")
    
#     # Filter valid samples only
#     valid_samples = filter_valid_samples(all_samples)
#     invalid_count = len(all_samples) - len(valid_samples)
#     print(f"Valid samples: {len(valid_samples)}")
#     print(f"Invalid samples filtered out: {invalid_count}")
    
#     # Calculate total unique documents
#     total_docs = set(sample.get(doc_id_key) for sample in valid_samples if sample.get(doc_id_key) is not None)
#     print(f"Total unique documents: {len(total_docs)}")
    
#     # Shuffle all valid samples
#     random.shuffle(valid_samples)
    
#     # Step 1: Randomly select test set (500 samples)
#     print(f"\nRandomly selecting {test_count} samples for test set...")
#     test_samples = valid_samples[:test_count]
#     remaining_samples = valid_samples[test_count:]
    
#     # Calculate document coverage for test
#     test_doc_ids = set(sample.get(doc_id_key) for sample in test_samples if sample.get(doc_id_key) is not None)
#     print(f"Test set: {len(test_samples)} samples from {len(test_doc_ids)} unique documents (out of {len(total_docs)} total documents)")
    
#     # Step 2: Remaining data split into train (80%) and dev (20%)
#     print(f"\nRemaining samples: {len(remaining_samples)}")
    
#     # Split: 80% train, 20% dev
#     train_size = int(len(remaining_samples) * 0.8)
#     train_samples = remaining_samples[:train_size]
#     dev_samples = remaining_samples[train_size:]
    
#     # Calculate unique documents covered for train and dev
#     train_doc_ids = set(sample.get(doc_id_key) for sample in train_samples if sample.get(doc_id_key) is not None)
#     dev_doc_ids = set(sample.get(doc_id_key) for sample in dev_samples if sample.get(doc_id_key) is not None)
    
#     print(f"Train set: {len(train_samples)} samples from {len(train_doc_ids)} unique documents (out of {len(total_docs)} total documents)")
#     print(f"Dev set: {len(dev_samples)} samples from {len(dev_doc_ids)} unique documents (out of {len(total_docs)} total documents)")
    
#     # Update serial numbers
#     for idx, sample in enumerate(test_samples, 1):
#         sample['serial-number'] = f"test-{idx}"
#     for idx, sample in enumerate(dev_samples, 1):
#         sample['serial-number'] = f"dev-{idx}"
#     for idx, sample in enumerate(train_samples, 1):
#         sample['serial-number'] = f"train-{idx}"
    
#     return {
#         'train': train_samples,
#         'dev': dev_samples,
#         'test': test_samples
#     }